In [ ]:
# =============================================================================
# Reinforcement Learning for Active Structural Vibration Control
# Algorithm: Twin Delayed Deep Deterministic Policy Gradient (TD3)
# =============================================================================
#
# Problem Description
# -------------------
# A 5-story shear building model is subjected to stochastic base excitation.
# The objective is to reduce structural vibration using active control forces.
#
# State Vector
# ------------
# [displacements, velocities, ground acceleration]
#
# Action
# ------
# Continuous control forces applied to each floor.
#
# Reward
# ------
# Penalizes large displacements and large control forces:
#
#   r = -( ||x||^2 + 0.01 ||u||^2 )
#
# where
#   x : displacement vector
#   u : control force vector
#
# The agent learns to minimize structural vibrations while keeping
# control effort small.
# =============================================================================



# =============================================================================
# 1. Install Dependencies (Colab)
# =============================================================================
!pip install stable-baselines3[extra] shimmy scipy --quiet



# =============================================================================
# 2. Import Libraries
# =============================================================================
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import StateSpace, lsim

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise



# =============================================================================
# 3. Custom Environment Definition
# =============================================================================
class VibrationControlEnv(gym.Env):

    """
    Reinforcement learning environment for active vibration control
    of a multi-story shear building.
    """

    def __init__(self):

        super().__init__()

        self.n_floors = 5
        self.mass = 5100
        self.k_val = 1.334e7

        self.dt = 0.01
        self.T = 20
        self.steps = int(self.T / self.dt)

        self.t = np.linspace(0, self.T, self.steps)

        # Structural matrices
        self.M = np.diag([self.mass] * self.n_floors)
        self.K = self._build_K()
        self.C = 0.02 * self.M + 0.001 * self.K

        # System matrices (computed once)
        self.A = np.block([
            [np.zeros((self.n_floors, self.n_floors)), np.eye(self.n_floors)],
            [-np.linalg.inv(self.M) @ self.K, -np.linalg.inv(self.M) @ self.C]
        ])

        self.B = np.block([
            [np.zeros((self.n_floors, self.n_floors))],
            [np.linalg.inv(self.M)]
        ])

        self.B_ext = np.block([
            [np.zeros((self.n_floors, 1))],
            [-np.ones((self.n_floors, 1))]
        ])

        self.sys = StateSpace(
            self.A,
            np.hstack([self.B, self.B_ext]),
            np.eye(2 * self.n_floors),
            np.zeros((2 * self.n_floors, self.n_floors + 1))
        )

        self.state = np.zeros(2 * self.n_floors)
        self.current_step = 0

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(2 * self.n_floors + 1,),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=-1000,
            high=1000,
            shape=(self.n_floors,),
            dtype=np.float32
        )


    def _build_K(self):

        K = np.zeros((self.n_floors, self.n_floors))

        for i in range(self.n_floors):

            if i > 0:
                K[i, i] += self.k_val
                K[i, i - 1] -= self.k_val
                K[i - 1, i] -= self.k_val
                K[i - 1, i - 1] += self.k_val
            else:
                K[i, i] += self.k_val

        return K


    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.state = np.zeros(2 * self.n_floors)
        self.current_step = 0

        self.input_signal = 0.05 * np.random.randn(self.steps)

        return self._get_obs(), {}


    def _get_obs(self):

        a_g = self.input_signal[self.current_step] if self.current_step < self.steps else 0

        return np.concatenate([self.state, [a_g]]).astype(np.float32)


    def step(self, action):

        if self.current_step >= self.steps:

            return self._get_obs(), 0.0, True, False, {}

        u = action
        a_g = self.input_signal[self.current_step]

        u_ext = np.hstack([u, a_g])

        _, y, _ = lsim(
            self.sys,
            U=np.tile(u_ext, (2, 1)),
            T=[0, self.dt],
            X0=self.state
        )

        self.state = y[-1]

        self.current_step += 1

        terminated = self.current_step >= self.steps
        truncated = False

        displacement = self.state[:self.n_floors]

        reward = -(
            np.linalg.norm(displacement)**2 +
            0.01 * np.linalg.norm(u)**2
        )

        return self._get_obs(), reward, terminated, truncated, {}



# =============================================================================
# 4. Environment Initialization
# =============================================================================
env = VibrationControlEnv()



# =============================================================================
# 5. TD3 Configuration
# =============================================================================
n_actions = env.action_space.shape[0]

action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=0.1 * np.ones(n_actions)
)

policy_kwargs = dict(
    net_arch=[256, 256],
    activation_fn=torch.nn.ReLU
)



# =============================================================================
# 6. Create TD3 Model
# =============================================================================
model = TD3(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    verbose=1,
    learning_rate=1e-4,
    policy_kwargs=policy_kwargs,
    batch_size=64,
    train_freq=1,
    gradient_steps=1,
    buffer_size=100000,
    learning_starts=1000
)



# =============================================================================
# 7. Train the Model
# =============================================================================
model.learn(total_timesteps=10000)



# =============================================================================
# 8. Simulation Function
# =============================================================================
def simulate_response(controlled=True, model=None):

    env = VibrationControlEnv()

    obs, _ = env.reset()

    responses = []

    for _ in range(env.steps):

        if controlled:
            action, _ = model.predict(obs, deterministic=True)
        else:
            action = np.zeros(env.n_floors)

        obs, _, _, _, _ = env.step(action)

        responses.append(env.state[:env.n_floors])

    return np.array(responses)



# =============================================================================
# 9. Run Simulations
# =============================================================================
response_no_control = simulate_response(controlled=False)

response_with_control = simulate_response(controlled=True, model=model)



# =============================================================================
# 10. Plot Structural Response
# =============================================================================
plt.figure(figsize=(12,6))

for i in range(5):

    plt.subplot(2,3,i+1)

    plt.plot(response_no_control[:,i],'--',label="No Control")

    plt.plot(response_with_control[:,i],'-',label="TD3 Control")

    plt.title(f"Floor {i+1}")

    plt.xlabel("Time Step")

    plt.ylabel("Displacement")

    plt.grid(True)

    plt.legend()

plt.tight_layout()

plt.suptitle("Structural Displacement With and Without Control", y=1.02)

plt.show()



# =============================================================================
# 11. Vibration Reduction Factor (VRF)
# =============================================================================
for i in range(5):

    r_nc = np.max(np.abs(response_no_control[:,i]))

    r_wc = np.max(np.abs(response_with_control[:,i]))

    vrf = 100*(r_nc - r_wc)/r_nc

    print(f"Floor {i+1} - Vibration Reduction Factor: {vrf:.2f}%")
